# Colab 2 — Add a Subgraph Directly

**Goal:** A compiled subgraph can behave like one node when parent and child share state keys.


In [1]:
!pip -q install -U langgraph


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 3.5 MB/s eta 0:00:00


In [2]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# Same Schema for PARENT and Subgraph
class State(TypedDict):
    request: str
    result: str

# ----- Product subgraph -----
def search_catalog(state: State):
    #actual work
    return {"result": f"Found headphones for: {state['request']}"}

product_builder = StateGraph(State)
product_builder.add_node("search_catalog", search_catalog)
product_builder.add_edge(START, "search_catalog")
product_builder.add_edge("search_catalog", END)
product_subgraph = product_builder.compile()

# ----- PARENT GRAPH -----
parent_builder = StateGraph(State)
parent_builder.add_node("product_agent", product_subgraph)
parent_builder.add_edge(START, "product_agent")
#more and more nodes
parent_builder.add_edge("product_agent", END)
parent_graph = parent_builder.compile()

parent_graph.invoke({"request": "wireless headphones under $300", "result": ""})


{'request': 'wireless headphones under $300',
 'result': 'Found headphones for: wireless headphones under $300'}

### Takeaway

> “The parent and subgraph both understand `request` and `result`, so we plug the compiled graph in directly—no translator.”
